Convert navigable rivers data from old text file format into a shapefile for uploading to Google Earth Engine 

In [2]:
import os
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

def get_txt_files(folder_path, substring="-riv"):
    """
    Return all .txt files in folder (except README.txt), 
    optionally filtering by a substring (default: '-riv').
    """
    return [
        f for f in os.listdir(folder_path)
        if f.lower().endswith('.txt')
        and f.lower() != 'readme.txt'
        and substring.lower() in f.lower()
    ]

def build_geodataframe(segments):
    features = []
    for seg in segments:
        # Only include segments with at least 2 coordinates
        if len(seg["coordinates"]) >= 2:
            line = LineString(seg["coordinates"])
            features.append({
                "segment_id": seg["segment_id"],
                "rank": seg["rank"],
                "geometry": line
            })
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def process_txt_to_gdf(folder_path, txt_file):
    """Parse a .txt file and return a GeoDataFrame."""
    file_path = os.path.join(folder_path, txt_file)
    segments = parse_segments_from_file(file_path)
    return build_geodataframe(segments)



    """Save a GeoDataFrame to a GeoJSON file."""
    gdf.to_file(output_path, driver="GeoJSON")

def process_folder_to_geojsons(folder_path, output_folder, combine_all=False, substring="-riv"):
    """
    Process all .txt files in folder matching the substring,
    save each as GeoJSON, optionally combine all into one GeoDataFrame and GeoJSON.
    """
    os.makedirs(output_folder, exist_ok=True)
    txt_files = get_txt_files(folder_path, substring)
    gdf_list = []
    for txt_file in txt_files:
        gdf = process_txt_to_gdf(folder_path, txt_file)
        gdf_list.append(gdf)
        geojson_path = os.path.join(output_folder, txt_file.replace('.txt', '.geojson'))
        save_gdf_to_geojson(gdf, geojson_path)
    combined_gdf = None
    if combine_all and gdf_list:
        combined_gdf = gpd.GeoDataFrame(pd.concat(gdf_list, ignore_index=True), crs=gdf_list[0].crs)
        combined_path = os.path.join(output_folder, "combined.geojson")
        save_gdf_to_geojson(combined_gdf, combined_path)
        return combined_path, combined_gdf
    return [os.path.join(output_folder, f.replace('.txt', '.geojson')) for f in txt_files], combined_gdf

def parse_segments_from_file(file_path):
    segments = []
    with open(file_path, 'r') as file:
        current_segment = None
        for line in file:
            line = line.strip()
            if line.startswith("segment"):
                if current_segment:
                    segments.append(current_segment)
                parts = line.split()
                current_segment = {
                    "segment_id": int(parts[1]),
                    "rank": int(parts[3]),
                    "coordinates": []
                }
            elif line:
                lon, lat = map(float, line.split())
                current_segment["coordinates"].append((lat, lon)) # Note: Shapely uses (lat, lon) order  
        if current_segment:
            segments.append(current_segment)
    return segments

def build_geodataframe(segments):
    features = []
    for seg in segments:
        # Only include segments with at least 2 coordinates
        if len(seg["coordinates"]) >= 2:
            line = LineString(seg["coordinates"])
            features.append({
                "segment_id": seg["segment_id"],
                "rank": seg["rank"],
                "geometry": line
            })
    return gpd.GeoDataFrame(features, crs="EPSG:4326")

def save_gdf_to_geojson(gdf, output_path):
    """Save a GeoDataFrame to a GeoJSON file, ensuring valid geometries and simple attributes."""
    # Remove empty or invalid geometries
    gdf = gdf[gdf.is_valid & ~gdf.is_empty]
    # Optionally fix invalid geometries (buffer(0) trick)
    gdf['geometry'] = gdf['geometry'].buffer(0)
    # Keep only simple columns
    gdf = gdf[['segment_id', 'rank', 'geometry']]
    # Save as GeoJSON with UTF-8 encoding
    gdf.to_file(output_path, driver="GeoJSON", encoding="utf-8")
    
#     return segments
# # Example usage:
# folder_path = r"C:\Users\Arnell\Downloads\WDB-text (2).tar\WDB"
# output_folder = os.path.join(folder_path, "outputs")
# geojson_files, combined_gdf = process_folder_to_geojsons(folder_path, output_folder, combine_all=True, substring="-riv")
# print("GeoJSON files saved:", geojson_files)
# if combined_gdf is not None:
#     print(combined_gdf.head())

In [3]:
# Example usage:
folder_path = r"C:\Users\Arnell\Downloads\WDB-text (2).tar\WDB"
output_folder = os.path.join(folder_path, "outputs")
geojson_files, combined_gdf = process_folder_to_geojsons(folder_path, output_folder, combine_all=True, substring="-riv")
print("GeoJSON files saved:", geojson_files)
if combined_gdf is not None:
    print(combined_gdf.head())

GeoJSON files saved: C:\Users\Arnell\Downloads\WDB-text (2).tar\WDB\outputs\combined.geojson
   segment_id  rank                                           geometry
0           1     1  LINESTRING (31.30583 27.09778, 31.30583 27.099...
1           2     1  LINESTRING (31.22944 29.38778, 31.22944 29.389...
2           3     1  LINESTRING (30.84889 30.54361, 30.84778 30.544...
3           4     3  LINESTRING (31.25194 27.165, 31.24972 27.16472...
4           5     3  LINESTRING (31.18194 27.19389, 31.18083 27.195...


In [4]:
print(combined_gdf.head())
print(combined_gdf.is_valid.all(), combined_gdf.is_empty.any())



   segment_id  rank                                           geometry
0           1     1  LINESTRING (31.30583 27.09778, 31.30583 27.099...
1           2     1  LINESTRING (31.22944 29.38778, 31.22944 29.389...
2           3     1  LINESTRING (30.84889 30.54361, 30.84778 30.544...
3           4     3  LINESTRING (31.25194 27.165, 31.24972 27.16472...
4           5     3  LINESTRING (31.18194 27.19389, 31.18083 27.195...
True False


In [ ]:
# import ee
# import geopandas as gpd
# import json
# ee.Authenticate()
# ee.Initialize(project='ee-andyarnellgee')  # Replace with your GEE cloud project ID



Successfully saved authorization token.


In [9]:
def gdf_to_ee_featurecollection(gdf):
    """
    Convert a GeoPandas GeoDataFrame to an Earth Engine FeatureCollection.
    """
    # Convert to GeoJSON dict
    geojson_dict = json.loads(gdf.to_json())
    # Convert to ee.FeatureCollection
    fc = ee.FeatureCollection(geojson_dict)
    return fc

# Example usage:
# combined_gdf = ... (your combined GeoDataFrame)
ee_fc = gdf_to_ee_featurecollection(combined_gdf.head(10))


In [10]:
print("Earth Engine FeatureCollectio size:", ee_fc.size().getInfo())

Earth Engine FeatureCollectio size: 10


In [11]:
import geemap
m = geemap.Map()
m.addLayer(ee_fc.limit(100), {}, "Rivers")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [12]:
print(combined_gdf.dtypes)
print(combined_gdf.head())
print(combined_gdf.is_valid.all(), combined_gdf.is_empty.any())
print(combined_gdf.columns)

segment_id       int64
rank             int64
geometry      geometry
dtype: object
   segment_id  rank                                           geometry
0           1     1  LINESTRING (31.30583 27.09778, 31.30583 27.099...
1           2     1  LINESTRING (31.22944 29.38778, 31.22944 29.389...
2           3     1  LINESTRING (30.84889 30.54361, 30.84778 30.544...
3           4     3  LINESTRING (31.25194 27.165, 31.24972 27.16472...
4           5     3  LINESTRING (31.18194 27.19389, 31.18083 27.195...
True False
Index(['segment_id', 'rank', 'geometry'], dtype='object')


In [13]:
single_feature_path = os.path.join(output_folder, "single_feature.geojson")
combined_gdf.iloc[[0]].to_file(single_feature_path, driver="GeoJSON", encoding="utf-8")

In [14]:
# Export combined_gdf to a shapefile
shapefile_path = os.path.join(output_folder, "combined_gdf.shp")
combined_gdf.to_file(shapefile_path, driver="ESRI Shapefile", encoding="utf-8")
print(f"Shapefile saved to: {shapefile_path}")

Shapefile saved to: C:\Users\Arnell\Downloads\WDB-text (2).tar\WDB\outputs\combined_gdf.shp
